In [11]:
!pip install -q -U transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 131.6 MB/s eta 0:00:00


In [32]:
import pandas as pd
import numpy as np
import regex as re
from sklearn.model_selection import train_test_split

In [33]:
from transformers import RobertaTokenizer, RobertaModel
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaModel.from_pretrained('roberta-base')

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
df = pd.read_csv("/content/drive/MyDrive/final_dataframe.csv")

def clean_text(x):
    x = re.sub(r'\s+', ' ', x)
    x = re.sub(r'http\S+', '', x)
    x = re.sub(r'[^A-Za-z0-9 .,]', '', x)
    return x.strip()

df["resume"] = df["resume"].apply(clean_text)
df["jd"] = df["jd"].apply(clean_text)
df["new_ats"] = df["new_ats"] / 100.0


In [35]:
X = df[["resume", "jd"]]
y = df['new_ats']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2 , random_state= 42)

In [36]:
train_encoding = tokenizer(
    X_train["resume"].tolist(),
    X_train["jd"].tolist(),
    padding=True,
    truncation=True,
    return_tensors="pt"
)

test_encoding = tokenizer(
    X_test["resume"].tolist(),
    X_test["jd"].tolist(),
    padding=True,
    truncation=True,
    return_tensors="pt"
)




Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

**Converting the embeding to a dataset object**

In [38]:
import torch
from torch.utils.data import Dataset, DataLoader

In [39]:
class ResumeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = ResumeDataset(train_encoding, y_train.reset_index(drop=True))
test_dataset  = ResumeDataset(test_encoding, y_test.reset_index(drop=True))


In [40]:
from transformers import RobertaForSequenceClassification, Trainer, TrainingArguments
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=1
)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [41]:
training_args = TrainingArguments(
    output_dir="./results",
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)



In [42]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze()
    mse = mean_squared_error(labels, preds)
    mae = mean_absolute_error(labels, preds)
    return {"mse": mse, "mae": mae}


In [43]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


In [20]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: p-gupta98156 (p-gupta98156-daviet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/tmp/ipython-input-581857994.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Step,Training Loss


TrainOutput(global_step=474, training_loss=0.02598729515880472, metrics={'train_runtime': 410.4885, 'train_samples_per_second': 9.209, 'train_steps_per_second': 1.155, 'total_flos': 994550859509760.0, 'train_loss': 0.02598729515880472, 'epoch': 3.0})

In [30]:
output_dir = "/content/drive/MyDrive/Project/ATS-Finetuned-final"

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ Fine-tuning complete. Model and tokenizer saved successfully!")


AttributeError: 'str' object has no attribute 'save_pretrained'

In [44]:
output_dir = "/content/drive/MyDrive/Project/roberto_base"

tokenizer = RobertaTokenizer.from_pretrained(output_dir)
model = RobertaForSequenceClassification.from_pretrained(output_dir)
model.eval()  # set to evaluation mode


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [46]:
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Make sure model is in eval mode
model.eval()

predicted_scores = []

# Iterate over test set
for index, row in X_test.iterrows():
    # Get the resume and JD text
    resume = row['resume']
    jd = row['jd']

    # Tokenize the pair
    encoding = tokenizer(
        resume,
        jd,
        padding='max_length',
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )

    with torch.no_grad():  # no gradients needed
        outputs = model(**encoding)
        score = outputs.logits.squeeze().item()  # get scalar
        predicted_scores.append(score)

# Convert to numpy array
predicted_scores = np.array(predicted_scores)

# Actual scores
actual_scores = y_test.values

# Metrics
mse = mean_squared_error(actual_scores, predicted_scores)
mae = mean_absolute_error(actual_scores, predicted_scores)
rmse = np.sqrt(mse)

print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}")

# --- Qualitative evaluation ---
results_df = X_test.copy()
results_df['actual_score'] = actual_scores
results_df['predicted_score'] = predicted_scores
results_df['error'] = abs(results_df['actual_score'] - results_df['predicted_score'])

print("\nTop 5 Best Predictions (Lowest Error):")
print(results_df.nsmallest(5, 'error'))

print("\nTop 5 Worst Predictions (Highest Error):")
print(results_df.nlargest(5, 'error'))


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

MSE: 0.0094, MAE: 0.0719, RMSE: 0.0969

Top 5 Best Predictions (Lowest Error):
                                                 resume  \
422   S A V I C H O P R A LINKEDIN GITHUB EMAIL SUMM...   
485   Machine Learning Engineer Data Analyst Profici...   
1350  Shaurya Soni Discipline Bachelor of Technology...   
494   Machine Learning Engineer Data Analyst Profici...   
1348  Shaurya Soni Discipline Bachelor of Technology...   

                                                     jd  actual_score  \
422   As a Full Stack Engineer at Prime Corporate, y...          0.78   
485   Full job description Job Summary We are seekin...          0.45   
1350  Software Tester Intern About Us We are an inno...          0.78   
494   As a Full Stack Engineer at Prime Corporate, y...          0.65   
1348  We are looking for a passionate and enthusiast...          0.85   

      predicted_score     error  
422          0.780190  0.000190  
485          0.449771  0.000229  
1350         0.779746  0.